# Eddy tilt and vertical velocity strength

Test whether more strongly tilted eddies have stronger vertical motion in the upper 1,000 m. This notebook reuses the cache made by notebook 02 and fixes the footprint at core fraction 1.5. The primary vertical-velocity strength measure is RMS `w`; peak absolute velocity and the upward-to-downward range are sensitivity measures. Signed core-mean `w` is analysed separately because opposing upward and downward cells can cancel. Positive model `w` is upward.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import statsmodels.formula.api as smf

HERE=Path.cwd()
if HERE.name!='vertical_velocity': raise RuntimeError('Launch from the vertical_velocity folder')
sys.path.insert(0,str(HERE.parent))
from vertical_velocity_tools import column_extrema
import seacofs_tilt_tools as tilt


In [ ]:
FRACTION=1.5
MAX_DEPTH_M=1000
MIN_CORE_CELLS=8
MIN_COVERAGE=0.7
MIN_EDDIES_PER_DEPTH=30
N_BOOT=2000
SEED=731
CACHE_PATH=Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/vertical_velocity_climatology/core_vertical_velocity_upper_1000m.parquet')


In [ ]:
raw=pd.read_parquet(CACHE_PATH)
needed={'Eddy','Day','Cyc','fraction','Depth','TiltDis','n_valid','coverage','w_mean','w_rms','w_max','w_min'}
missing=needed-set(raw.columns)
if missing: raise ValueError(f'Cache is missing {sorted(missing)}; rebuild it with notebook 02')
if not np.isclose(raw.fraction,FRACTION).any(): raise ValueError(f'Fraction {FRACTION} absent from cache')
depth=raw.loc[np.isclose(raw.fraction,FRACTION)&raw.Depth.between(0,MAX_DEPTH_M)].copy()
depth=depth.loc[depth.n_valid.ge(MIN_CORE_CELLS)&depth.coverage.ge(MIN_COVERAGE)]
depth=depth.dropna(subset=['TiltDis','w_mean','w_rms','w_max','w_min'])
if depth.duplicated(['Eddy','Day','Depth']).any(): raise ValueError('Duplicate eddy-day-depth rows')
depth['w_peak_abs']=depth[['w_max','w_min']].abs().max(axis=1)
depth['w_range']=depth.w_max-depth.w_min
depth['abs_w_mean']=depth.w_mean.abs()
print(depth.groupby('Cyc').agg(eddies=('Eddy','nunique'),eddy_days=('Day','size'),depth_rows=('Depth','size'),median_coverage=('coverage','median')))


## Snapshot and eddy tables

The snapshot table gives one upper-1,000 m summary per eddy-day. RMS is weighted by valid sampled cells across fitted depths, matching notebook 02; it is not a volume-weighted vertical integral. The primary population table then averages snapshots within each eddy, so tracks rather than days receive equal weight.


In [ ]:
snapshot=column_extrema(depth)
identity=depth[['Eddy','Day','Cyc','TiltDis']].drop_duplicates(['Eddy','Day'])
snapshot=snapshot.merge(identity,on=['Eddy','Day'],how='left',validate='one_to_one')
snapshot['w_peak_abs']=snapshot[['w_max','w_min']].abs().max(axis=1)
snapshot['w_range']=snapshot.w_max-snapshot.w_min
snapshot['abs_w_mean']=snapshot.w_mean.abs()
surface,_=tilt.load_tilt_tables()
context_cols=[c for c in ['Eddy','Day','yc','Rc','Date','Region'] if c in surface]
context=surface[context_cols].drop_duplicates(['Eddy','Day'])
snapshot=snapshot.merge(context,on=['Eddy','Day'],how='left',validate='one_to_one')
eddy_cols=['TiltDis','w_rms','w_peak_abs','w_range','abs_w_mean','w_mean']
extra=[c for c in ['yc','Rc'] if c in snapshot]
eddy=snapshot.groupby(['Eddy','Cyc'],as_index=False)[eddy_cols+extra].mean()
print(snapshot.groupby('Cyc').agg(eddies=('Eddy','nunique'),snapshots=('Day','size')))
display(eddy.groupby('Cyc')[eddy_cols].median())


## Upper 1000 m relationship

Each point below is one eddy. Binned curves show median vertical-velocity strength within tilt quintiles. Spearman correlations are calculated separately for AEs and CEs and for the pooled population. Whole-eddy bootstrap intervals quantify uncertainty.


In [ ]:
def boot_spearman(frame,x,y,n_boot=N_BOOT,seed=SEED):
    d=frame[['Eddy',x,y]].dropna()
    ids=d.Eddy.unique()
    if len(ids)<4:return (np.nan,np.nan,np.nan,len(ids))
    est=spearmanr(d[x],d[y]).statistic
    rng=np.random.default_rng(seed); vals=[]
    for _ in range(n_boot):
        chosen=rng.choice(ids,len(ids),replace=True)
        b=pd.concat([d[d.Eddy.eq(e)] for e in chosen],ignore_index=True)
        vals.append(spearmanr(b[x],b[y]).statistic)
    lo,hi=np.nanquantile(vals,[.025,.975])
    return est,lo,hi,len(ids)

def binned(frame,x,y,bins=5):
    d=frame[[x,y]].dropna().copy()
    d['bin']=pd.qcut(d[x],min(bins,d[x].nunique()),duplicates='drop')
    return d.groupby('bin',observed=True).agg(x=(x,'median'),y=(y,'median'),q1=(y,lambda z:z.quantile(.25)),q3=(y,lambda z:z.quantile(.75)),n=(y,'size')).reset_index()

metrics={'w_rms':'RMS w','w_peak_abs':'Peak |w|','w_range':'Up–down range','abs_w_mean':'|signed mean w|'}
fig,axes=plt.subplots(2,2,figsize=(13,10))
results=[]
for ax,(metric,label) in zip(axes.flat,metrics.items()):
    for cyc,color in [('AE','tab:red'),('CE','tab:blue')]:
        d=eddy[eddy.Cyc.eq(cyc)]
        ax.scatter(d.TiltDis,1e3*d[metric],s=15,alpha=.2,color=color)
        curve=binned(d,'TiltDis',metric)
        ax.plot(curve.x,1e3*curve.y,'o-',color=color,label=cyc)
        est,lo,hi,n=boot_spearman(d,'TiltDis',metric)
        results.append(dict(scope=cyc,metric=metric,rho=est,lo=lo,hi=hi,n_eddies=n))
    est,lo,hi,n=boot_spearman(eddy,'TiltDis',metric)
    results.append(dict(scope='pooled',metric=metric,rho=est,lo=lo,hi=hi,n_eddies=n))
    ax.set(xlabel='Eddy-mean tilt magnitude (km)',ylabel=f'{label} (mm/s)',title=label)
    ax.legend()
fig.tight_layout();plt.show()
results=pd.DataFrame(results)
display(results)


## Weakly and strongly tilted eddies

For a direct analogue of the published tilted-versus-non-tilted comparison, this descriptive sensitivity test compares the lowest and highest tilt quartiles within each polarity. The categories are relative to this sampled population; the low group is not literally untilted.


In [ ]:
quartile_rows=[]
fig,axes=plt.subplots(1,3,figsize=(14,4),sharey=False)
for ax,metric in zip(axes,['w_rms','w_peak_abs','w_range']):
    positions=[];values=[];labels=[];colors=[];pos=1
    for cyc,color in [('AE','tab:red'),('CE','tab:blue')]:
        g=eddy[eddy.Cyc.eq(cyc)].dropna(subset=['TiltDis',metric]).copy()
        lo_q,hi_q=g.TiltDis.quantile([.25,.75])
        low=g.loc[g.TiltDis.le(lo_q),metric];high=g.loc[g.TiltDis.ge(hi_q),metric]
        rng=np.random.default_rng(SEED);draws=[]
        for _ in range(N_BOOT):
            draws.append(rng.choice(high,len(high),replace=True).mean()-rng.choice(low,len(low),replace=True).mean())
        ci=np.quantile(draws,[.025,.975])
        quartile_rows.append(dict(Cyc=cyc,metric=metric,low_tilt_max=lo_q,high_tilt_min=hi_q,
                                  low_mean=low.mean(),high_mean=high.mean(),difference=high.mean()-low.mean(),lo=ci[0],hi=ci[1],
                                  low_n=len(low),high_n=len(high)))
        values += [1e3*low,1e3*high];positions += [pos,pos+1];labels += [f'{cyc} low',f'{cyc} high'];colors += [color,color];pos+=3
    bp=ax.boxplot(values,positions=positions,tick_labels=labels,showfliers=False,patch_artist=True)
    for patch,color in zip(bp['boxes'],colors):patch.set_facecolor(color);patch.set_alpha(.35)
    ax.set(title=metrics[metric],ylabel='Vertical velocity strength (mm/s)')
    ax.tick_params(axis='x',rotation=25)
fig.tight_layout();plt.show()
display(pd.DataFrame(quartile_rows))


## Depth-resolved relationship

At each fitted depth, days are averaged within each eddy before calculating a correlation. A positive coefficient means more tilted eddies have stronger vertical motion at that depth. Depths with fewer than the required number of eddies in either polarity are omitted.


In [ ]:
depth_eddy=(depth.groupby(['Eddy','Cyc','Depth'],as_index=False)
            .agg(TiltDis=('TiltDis','mean'),w_rms=('w_rms','mean'),w_peak_abs=('w_peak_abs','mean'),
                 w_range=('w_range','mean'),abs_w_mean=('abs_w_mean','mean')))
depth_results=[]
for (cyc,z),g in depth_eddy.groupby(['Cyc','Depth']):
    if g.Eddy.nunique()<MIN_EDDIES_PER_DEPTH:continue
    for metric in metrics:
        est,lo,hi,n=boot_spearman(g,'TiltDis',metric,n_boot=1000,seed=SEED+int(round(z)))
        depth_results.append(dict(Cyc=cyc,Depth=z,metric=metric,rho=est,lo=lo,hi=hi,n_eddies=n))
depth_results=pd.DataFrame(depth_results)
fig,axes=plt.subplots(1,3,figsize=(15,6),sharey=True)
for ax,metric in zip(axes,['w_rms','w_peak_abs','w_range']):
    for cyc,color in [('AE','tab:red'),('CE','tab:blue')]:
        g=depth_results.query('metric==@metric and Cyc==@cyc').sort_values('Depth')
        ax.plot(g.rho,g.Depth,'o-',color=color,label=cyc)
        ax.fill_betweenx(g.Depth,g.lo,g.hi,color=color,alpha=.15)
    ax.axvline(0,color='0.4',lw=.8);ax.set(xlabel='Spearman correlation',title=metrics[metric]);ax.legend()
axes[0].set_ylabel('Fitted depth (m)');axes[0].invert_yaxis();fig.tight_layout();plt.show()


## Within-eddy relationship

The eddy-level analysis can reflect persistent differences between eddies. This section removes each eddy's mean tilt and mean vertical-velocity strength, then asks whether days when an eddy is more tilted than usual are also days when its vertical motion is stronger than usual. Eddies with fewer than two cached snapshots do not contribute.


In [ ]:
within=snapshot.groupby('Eddy').filter(lambda g:len(g)>=2).copy()
within['tilt_anom']=within.TiltDis-within.groupby('Eddy').TiltDis.transform('mean')
within_results=[]
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,metric in zip(axes,['w_rms','w_peak_abs','w_range']):
    within[f'{metric}_anom']=within[metric]-within.groupby('Eddy')[metric].transform('mean')
    for cyc,color in [('AE','tab:red'),('CE','tab:blue')]:
        d=within[within.Cyc.eq(cyc)]
        rho=spearmanr(d.tilt_anom,d[f'{metric}_anom']).statistic
        # Cluster bootstrap retains all days belonging to each resampled eddy.
        rng=np.random.default_rng(SEED);ids=d.Eddy.unique();boots=[]
        for _ in range(N_BOOT):
            b=pd.concat([d[d.Eddy.eq(e)] for e in rng.choice(ids,len(ids),replace=True)],ignore_index=True)
            boots.append(spearmanr(b.tilt_anom,b[f'{metric}_anom']).statistic)
        lo,hi=np.nanquantile(boots,[.025,.975])
        within_results.append(dict(Cyc=cyc,metric=metric,rho=rho,lo=lo,hi=hi,n_eddies=len(ids),n_days=len(d)))
        ax.scatter(d.tilt_anom,1e3*d[f'{metric}_anom'],s=10,alpha=.18,color=color,label=cyc)
    ax.axhline(0,color='.6',lw=.7);ax.axvline(0,color='.6',lw=.7)
    ax.set(xlabel='Tilt anomaly within eddy (km)',ylabel=f'{metrics[metric]} anomaly (mm/s)',title=metrics[metric]);ax.legend()
fig.tight_layout();plt.show()
display(pd.DataFrame(within_results))


## Adjusted eddy-level model

This sensitivity model estimates the relationship between log tilt and log RMS vertical velocity while allowing AE and CE intercepts and slopes to differ. Available northing and radius terms reduce broad geographic and size confounding. It remains observational: background strain, stratification, season, topography and interactions can influence both tilt and vertical velocity.


In [ ]:
model_data=eddy.loc[(eddy.TiltDis>0)&(eddy.w_rms>0)].copy()
model_data['log_tilt']=np.log(model_data.TiltDis)
model_data['log_w_rms']=np.log(model_data.w_rms)
terms=['log_tilt*C(Cyc)']
for c in ('yc','Rc'):
    if c in model_data and model_data[c].notna().sum()==len(model_data):
        model_data[f'{c}_z']=(model_data[c]-model_data[c].mean())/model_data[c].std()
        terms.append(f'{c}_z')
model=smf.ols('log_w_rms ~ '+' + '.join(terms),data=model_data).fit(cov_type='HC3')
display(model.summary2().tables[1])


## Interpretation rules

Evidence for the proposed link is strongest if RMS `w` increases with tilt in both polarities, the depth-resolved relationship spans a coherent depth range, and the within-eddy relationship has the same sign. A relationship confined to peak extrema is less robust because extrema depend on grid noise and the number of sampled cells. A between-eddy relationship without a within-eddy relationship means strongly tilted eddies differ systematically from weakly tilted eddies, but does not show that an individual eddy's vertical motion strengthens as its tilt changes.
